In [0]:
%sql
create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.arcaderisk_release_scope
as
select MI_Table_lists_ as mi_table, 
  case when Is_this_impacted_by_CIBT_W2_ in ('Y', 'P') then 'Y' else 'N' end as Impacted_Flag,
  case when Expected_Release_Cycle rlike '\\d' then 'Y' else 'N' end as Release_scheduled,
  Expected_Release_Cycle,
  Is_this_impacted_by_CIBT_W2_
from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.arcaderisk_proj_impact

In [0]:
%sql

create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.arcaderisk_project_impact_report
as   
WITH base AS (
  SELECT DISTINCT 
    fl1.cluster, 
    fl1.Document_Id,  
    upper(trim(c1.Impacted_Flag)) as Impacted_Flag, 
    upper(trim(c1.Release_scheduled)) as Release_scheduled, 
    CASE 
        WHEN upper(trim(c1.Impacted_Flag)) = 'Y' AND upper(trim(c1.Release_scheduled)) = 'Y' THEN 'Y'
        WHEN upper(trim(c1.Impacted_Flag)) = 'Y' AND upper(trim(c1.Release_scheduled)) = 'N' THEN 'N'
        ELSE null 
    END AS Prioritized,
    CASE WHEN upper(trim(c1.Impacted_Flag)) = 'N' OR upper(trim(c1.Impacted_Flag)) IS NULL THEN 'N' ELSE 'Y' END AS Impacted
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage AS fl1
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.arcaderisk_release_scope AS c1
    ON upper(trim(fl1.Calc_source_table)) = upper(trim(c1.mi_table))
  -- WHERE fl1.cluster IS NOT NULL
),
ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY Document_Id
      ORDER BY 
        Impacted DESC,     -- Y wins over N (keep Impacted=Y)
        CASE 
          WHEN Prioritized = 'N' THEN 1
          WHEN Prioritized = 'Y' THEN 2
          ELSE 3
        END ASC           -- N wins over Y, then null
    ) AS rn
  FROM base
)
select *, current_timestamp() as ingestion_ts from ranked
where rn = 1
;



In [0]:
%sql
create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.arcaderisk_project_impact_cluster
as   
WITH base AS (
  SELECT DISTINCT 
    fl1.cluster, 
    fl1.Document_Id,  
    upper(trim(c1.Impacted_Flag)) as Impacted_Flag, 
    upper(trim(c1.Release_scheduled)) as Release_scheduled, 
    CASE 
        WHEN upper(trim(c1.Impacted_Flag)) = 'Y' AND upper(trim(c1.Release_scheduled)) = 'Y' THEN 'Y'
        WHEN upper(trim(c1.Impacted_Flag)) = 'Y' AND upper(trim(c1.Release_scheduled)) = 'N' THEN 'N'
        ELSE null 
    END AS Prioritized,
    CASE WHEN upper(trim(c1.Impacted_Flag)) = 'N' OR upper(trim(c1.Impacted_Flag)) IS NULL THEN 'N' ELSE 'Y' END AS Impacted
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage AS fl1
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.arcaderisk_release_scope AS c1
    ON upper(trim(fl1.Calc_source_table)) = upper(trim(c1.mi_table))
  -- WHERE fl1.cluster IS NOT NULL
),
ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY Document_Id
      ORDER BY 
        Impacted DESC,     -- Y wins over N (keep Impacted=Y)
        CASE 
          WHEN Prioritized = 'N' THEN 1
          WHEN Prioritized = 'Y' THEN 2
          ELSE 3
        END ASC           -- N wins over Y, then null
    ) AS rn
  FROM base
),
deduped AS (
  SELECT cluster, Document_Id, Impacted, Prioritized
  FROM ranked
  WHERE rn = 1
)
SELECT 
  cluster,
  COUNT(DISTINCT CASE WHEN Impacted = 'Y' THEN Document_Id END) AS Impact_Y,
  COUNT(DISTINCT CASE WHEN Impacted = 'N' THEN Document_Id END) AS Impact_N,
  COUNT(DISTINCT CASE WHEN Prioritized = 'Y' THEN Document_Id END) AS Prioritized_Y,
  COUNT(DISTINCT CASE WHEN Prioritized = 'N' THEN Document_Id END) AS Prioritized_N,
  COUNT(DISTINCT CASE WHEN Prioritized is null THEN Document_Id END) AS No_Need_Prioritize,
  current_date() as ingestion_date
FROM deduped
GROUP BY cluster
ORDER BY cluster ASC

In [0]:
%sql
select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.arcaderisk_release_scope



select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.arcaderisk_project_impact_cluster

select 
  format_number(count(distinct case when Impacted="Y" and Prioritized="Y" then Document_Id end ), 0) as GREEN, 
  format_number(count(distinct case when Impacted="N" then Document_Id end ), 0) as GRAY,
  format_number(count(distinct case when Impacted="Y" and Prioritized="N" then Document_Id end ), 0) as RED 
from dataplatform01_central_dev_catalog_01.custom_sap_bo.arcaderisk_project_impact_report


select count(distinct document_id ) from dataplatform01_central_dev_catalog_01.custom_sap_bo.arcaderisk_project_impact_report



In [0]:
%sql
    
create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.arcaderisk_release_list
as
with exploded as (
  select distinct
    MI_Table_lists_ as mi_table, 
    case when Is_this_impacted_by_CIBT_W2_ in ('Y', 'P') then 'Y' else 'N' end as Impacted_Flag,
    Release_schedule,
    size(regexp_extract_all(Expected_Release_Cycle, '(Rel[ea]?ase\\s*\\d+)')) as Release_count,
    Is_this_impacted_by_CIBT_W2_
  from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.arcaderisk_proj_impact
    lateral view outer explode(regexp_extract_all(Expected_Release_Cycle, '(Rel[ea]?ase\\s*\\d+)')) t as Release_schedule
),
ranked as (
  select *,
    row_number() over (partition by mi_table order by cast(regexp_extract(Release_schedule, '\\d+', 0) as int) desc) as Release_rank,
    first_value(Release_schedule) over (partition by mi_table order by cast(regexp_extract(Release_schedule, '\\d+', 0) as int) desc) as Highest_release
  from exploded
)
select mi_table,
Impacted_Flag, 
Release_schedule
 from ranked
where Release_rank = 1

In [0]:
%sql
    
create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.arcaderisk_release_impact_report
WITH base AS (
  SELECT DISTINCT 
    fl1.cluster, 
    fl1.Document_Id,  
    upper(trim(c1.Impacted_Flag)) as Impacted_Flag, 
    upper(trim(c1.Release_schedule)) as Release_schedule, 
    CASE 
        WHEN upper(trim(c1.Impacted_Flag)) = 'Y' AND c1.Release_schedule is not null THEN 'Y'
        WHEN upper(trim(c1.Impacted_Flag)) = 'Y' AND c1.Release_schedule is null THEN 'N'
        ELSE null 
    END AS Prioritized,
    CASE WHEN upper(trim(c1.Impacted_Flag)) = 'N' OR upper(trim(c1.Impacted_Flag)) IS NULL THEN 'N' ELSE 'Y' END AS Impacted
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage AS fl1
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.arcaderisk_release_list AS c1
    ON upper(trim(fl1.Calc_source_table)) = upper(trim(c1.mi_table))
  -- WHERE fl1.cluster IS NOT NULL
),
ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY Document_Id
      ORDER BY 
        Impacted DESC,     -- Y wins over N (keep Impacted=Y)
        CASE 
          WHEN Prioritized = 'N' THEN 1
          WHEN Prioritized = 'Y' THEN 2
          ELSE 3
        END ASC,           -- N wins over Y, then null
        cast(regexp_extract(Release_schedule, '\\d+', 0) as int) DESC  -- higher release number wins
    ) AS rn
  FROM base
)
select *, current_timestamp() as ingestion_ts from ranked
where rn = 1
;
--   SELECT *,
--     ROW_NUMBER() OVER (
--       PARTITION BY Document_Id
--       ORDER BY 
--         Impacted DESC,     -- Y wins over N (keep Impacted=Y)
--         CASE 
--           WHEN Prioritized = 'N' THEN 1
--           WHEN Prioritized = 'Y' THEN 2
--           ELSE 3
--         END ASC,           -- N wins over Y, then null
--         cast(regexp_extract(Release_schedule, '\\d+', 0) as int) DESC  -- higher release number wins
--     ) AS rn
--   FROM base
-- order by Document_Id asc, rn asc


-- where impacted='Y'
-- and prioritized='N'
-- and rn > 1
-- -- and Release_schedule is null
-- -- ,
-- -- ranked AS (
-- --   SELECT *,
-- --     ROW_NUMBER() OVER (
-- --       PARTITION BY Document_Id
-- --       ORDER BY 
-- --         Impacted DESC,     -- Y wins over N (keep Impacted=Y)
-- --         CASE 
-- --           WHEN Prioritized = 'N' THEN 1
-- --           WHEN Prioritized = 'Y' THEN 2
-- --           ELSE 3
-- --         END ASC           -- N wins over Y, then null
-- --     ) AS rn
-- --   FROM base
-- -- )
-- -- select *, current_timestamp() as ingestion_ts from ranked
-- -- where rn = 1
-- -- ;

In [0]:
%sql
    
select * from (
  select Impacted, Release_schedule, Document_Id
  from dataplatform01_central_dev_catalog_01.custom_sap_bo.arcaderisk_release_impact_report
)
PIVOT (
  count(distinct Document_Id)
  FOR Release_schedule IN (
    'RELEASE 2' as Release_2,
    'RELEASE 4' as Release_4,
    'RELASE 5' as Relase_5,
    'RELEASE 5' as Release_5,
    'RELEASE 6' as Release_6,
    'RELEASE 7' as Release_7,
    'RELEASE 11' as Release_11,
    NULL as No_Release
  )
)
order by Impacted asc